pylidc looks in your home folder for a configuration file called, .pylidcrc on Mac and Linux, or pylidc.conf on Windows. You must create this file. On Linux and Mac, the file should be located at /home/[user]/.pylidcrc. On Windows, the file should be located at C:Users[User]pylidc.conf.

The configuration file should be formatted as follows:

```bash
[dicom]
path = /path/to/big_external_drive/datasets/LIDC-IDRI
warn = True
```

### After data cleaning:

In [13]:
import pylidc as pl
print(pl.__version__)

0.2.3


## LIDC-IDRI preprocessing

tldr: 
1. group the annotations
2. convert to hu
3. optional: clipping
4. create the nodule masks
5. resampling
---

A scan can have several annotations, but which ones refer to the same nodule? This is estimated using `cluster_annotations()`, which groups annotations based on the distance between them. Problems with this grouping will be analysed in a separate notebook ([PyLIDC issue #18](https://github.com/notmatthancock/pylidc/issues/18)).

After grouping the annotations, the next preprocessing step is to obtain each CT scan as a 3D array in Hounsfield Units (HU), using `scan.to_volume()` ([Wang et al., Section 2.2.4.3.1](https://pmc.ncbi.nlm.nih.gov/articles/PMC10361226/)). Since CT intensities are quantitative, we expect that keeping the original HU values without normalization may be the best option ([Santinha et al.](https://pmc.ncbi.nlm.nih.gov/articles/PMC11835989/)). However, we will also test clipping the values to `[-1000, 400]` HU, where `-1000` represents air and values around `400` begin to include denser bone and calcified structures. This may focus the image values on tissues relevant to lung nodules and help radiomic feature extraction. [Sousa et al.](https://www.mdpi.com/2076-3417/12/4/1959) used this interval before training their neural network.

The segmentations of the same nodule made by different radiologists are then combined into one mask using `consensus()`. Different consensus levels (`clevel`) will be tested to compare different ideas we have about what's best or not... we also have to take into consideration what this guy says here: 

"There are a number of preprocessing and data curation steps which are considered fixed when using the LIDC and almost all recent deep learning papers
follow them. These include (1) retaining only nodules that have been annotated
by at least three radiologists and (2) discarding nodules annotated as indeterminate." 

https://arxiv.org/pdf/2108.05386 problem with teh datasets studies

check this!!! 

In [14]:
#scans = pl.query(pl.Scan)
scans = pl.query(pl.Scan).limit(5)
print(f"{scans.count()} scans were selected")

5 scans were selected


In [15]:
from pylidc.utils import consensus

for scan in scans:
    nods = scan.cluster_annotations()
    vol = scan.to_volume(verbose=False)

    for anns in nods:
        # Perform a consensus consolidation and 50% agreement level.
        cmask,cbbox,masks = consensus(anns, clevel=0.5)

Different acquisition protocols can produce different radiomic features, partly because CT scans have different pixel and slice spacings. Therefore, the images and masks will be resampled to a common voxel spacing to make the extracted features more comparable and reproducible ([Wang et al., Section 2.2.4.3.2](https://pmc.ncbi.nlm.nih.gov/articles/PMC10361226/); [Santinha et al.](https://pmc.ncbi.nlm.nih.gov/articles/PMC11835989/)). For radiomics, this will initially be performed through [PyRadiomics](https://pyradiomics.readthedocs.io/en/latest/customization.html#resampling-the-image).
The CNN used later to generate deep features will use its own resampling step.

### Radiologist ratings

Only malignancy is used to create the prediction targets. Its aggregation and the target definitions are analysed in [malignancy_eda_splits_lidc_lndb.ipynb](malignancy_eda_splits_lidc_lndb.ipynb).